In [7]:
import os
import sys
import urllib.request
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 1. DYNAMICALLY FETCH THE OFFICIAL 345 CLASS ROSTER
print("Retrieving the official 345-class QuickDraw roster...")
categories_url = "https://raw.githubusercontent.com/googlecreativelab/quickdraw-dataset/master/categories.txt"
response = urllib.request.urlopen(categories_url)
official_categories = [line.decode('utf-8').strip().replace(' ', '_') for line in response.readlines()]

categories = sorted(list(set(official_categories)))
num_classes = len(categories)
samples_per_class = 5000
local_download_dir = '/content/quickdraw_data'
os.makedirs(local_download_dir, exist_ok=True)
base_url = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

print(f"Target count verified successfully: {num_classes} classes.")
print("Auditing storage partitions for true sample capacity requirements...")

# 2. AUDIT CAPACITY WINDOWS AND REPAIR TRUNCATIONS IN REAL TIME
for index, category in enumerate(categories):
    cloud_name = category.replace('_', '%20')
    file_name = f"{category}.npy"
    url = f"{base_url}{cloud_name}.npy"
    save_path = os.path.join(local_download_dir, file_name)

    if os.path.exists(save_path):
        try:
            # Open file pointer in read-only memmap mode to inspect the inner shape array
            temp_check = np.load(save_path, mmap_mode='r')
            available_samples = temp_check.shape[0]

            # If the file physically has fewer than 5000 rows, it is truncated
            if available_samples < samples_per_class:
                print(f"⚠️ Truncation found: '{category}' has only {available_samples} samples. Purging file...")
                del temp_check # Release file pointer lookups
                os.remove(save_path)
        except Exception as e:
            print(f"⚠️ Unreadable array format detected for '{category}'. Purging file... Error: {e}")
            if 'temp_check' in locals(): del temp_check
            try: os.remove(save_path)
            except: pass

    # If the file was missing or thrown out by the validation test, trigger a repair download
    if not os.path.exists(save_path):
        print(f"📥 [{index + 1}/{num_classes}] Fetching fresh backup stream: {file_name}")
        try:
            urllib.request.urlretrieve(url, save_path)
        except Exception as e:
            print(f"Critical error downloading clean array for {category}: {e}")

print("✅ Data Integrity Verified: Every partition matches baseline sample density rules.")

# 3. METADATA PARTITIONING
train_samples_per_class = int(samples_per_class * 0.75)
val_samples_per_class = samples_per_class - train_samples_per_class

train_meta = []
val_meta = []

for class_idx, category in enumerate(categories):
    for s_idx in range(train_samples_per_class):
        train_meta.append((class_idx, s_idx))
    for s_idx in range(train_samples_per_class, samples_per_class):
        val_meta.append((class_idx, s_idx))

train_meta = np.array(train_meta)
val_meta = np.array(val_meta)
np.random.shuffle(train_meta)
np.random.shuffle(val_meta)

# Memory-map verified data blocks directly on disk
print("Initial memory-mapping pointers across clean file structures...")
mmaps = {
    idx: np.load(os.path.join(local_download_dir, f"{cat}.npy"), mmap_mode='r')
    for idx, cat in enumerate(categories)
}
print("Pointers mapped without structural dimension conflicts.")

# 4. GLOBAL DATASTREAM GENERATOR
def shuffled_generator(meta_records, mmap_dict, batch_size=128):
    while True:
        indices = np.arange(len(meta_records))
        np.random.shuffle(indices)

        X_batch, y_batch = [], []
        for idx in indices:
            class_idx, sample_idx = meta_records[idx]
            sample = mmap_dict[class_idx][sample_idx]

            X_batch.append(sample.reshape(28, 28, 1).astype('float32') / 255.0)
            y_batch.append(tf.keras.utils.to_categorical(class_idx, num_classes=num_classes))

            if len(X_batch) == batch_size:
                yield np.array(X_batch), np.array(y_batch)
                X_batch, y_batch = [], []

# 5. STREAM TUNING CONFIGURATION
batch_size = 128
train_steps = len(train_meta) // batch_size
val_steps = len(val_meta) // batch_size

train_dataset = tf.data.Dataset.from_generator(
    lambda: shuffled_generator(train_meta, mmaps, batch_size),
    output_signature=(
        tf.TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: shuffled_generator(val_meta, mmaps, batch_size),
    output_signature=(
        tf.TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(None, num_classes), dtype=tf.float32)
    )
).prefetch(tf.data.AUTOTUNE)

print("Streaming pipelines generated successfully.")

# 6. ENFORCE CONTROLLED EXPERIMENT CNN DESIGN
model = Sequential([
    Conv2D(30, (3, 3), activation='relu', input_shape=(28, 28, 1), name="Conv_Block_1"),
    MaxPooling2D((2, 2), name="MaxPool_1"),
    Conv2D(15, (3, 3), activation='relu', name="Conv_Block_2"),
    MaxPooling2D((2, 2), name="MaxPool_2"),
    Dropout(0.2, name="Overfitting_Guard_Dropout"),
    Flatten(name="Vector_Flatten"),
    Dense(128, activation='relu', name="Dense_Hidden_1"),
    Dense(50, activation='relu', name="Dense_Hidden_2"),
    Dense(num_classes, activation='softmax', name="Softmax_Output")
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint = ModelCheckpoint(
    "sketchxai_baseline_345_model.h5",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

print("Starting Baseline Training pass...")
history = model.fit(
    train_dataset,
    steps_per_epoch=train_steps,
    validation_data=val_dataset,
    validation_steps=val_steps,
    epochs=12,
    callbacks=[checkpoint, early_stopping],
    verbose=1
)

Retrieving the official 345-class QuickDraw roster...
Target count verified successfully: 345 classes.
Auditing storage partitions for true sample capacity requirements...
⚠️ Unreadable array format detected for 'toilet'. Purging file... Error: mmap length is greater than file size
📥 [316/345] Fetching fresh backup stream: toilet.npy
✅ Data Integrity Verified: Every partition matches baseline sample density rules.
Initial memory-mapping pointers across clean file structures...
Pointers mapped without structural dimension conflicts.
Streaming pipelines generated successfully.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Starting Baseline Training pass...
Epoch 1/12
10106/10107 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2622 - loss: 3.5218
Epoch 1: val_accuracy improved from None to 0.45526, saving model to sketchxai_baseline_345_model.h5



Epoch 1: finished saving model to sketchxai_baseline_345_model.h5
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 147s 14ms/step - accuracy: 0.3472 - loss: 2.9740 - val_accuracy: 0.4553 - val_loss: 2.3846
Epoch 2/12
10106/10107 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4322 - loss: 2.4883
Epoch 2: val_accuracy improved from 0.45526 to 0.49332, saving model to sketchxai_baseline_345_model.h5



Epoch 2: finished saving model to sketchxai_baseline_345_model.h5
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 128s 13ms/step - accuracy: 0.4410 - loss: 2.4472 - val_accuracy: 0.4933 - val_loss: 2.2030
Epoch 3/12
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4612 - loss: 2.3463
Epoch 3: val_accuracy improved from 0.49332 to 0.50976, saving model to sketchxai_baseline_345_model.h5


...


Epoch 7: finished saving model to sketchxai_baseline_345_model.h5
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 139s 14ms/step - accuracy: 0.4967 - loss: 2.1623 - val_accuracy: 0.5378 - val_loss: 1.9853
Epoch 8/12
10103/10107 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5004 - loss: 2.1467
Epoch 8: val_accuracy improved from 0.53778 to 0.53873, saving model to sketchxai_baseline_345_model.h5



Epoch 8: finished saving model to sketchxai_baseline_345_model.h5
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 130s 13ms/step - accuracy: 0.5001 - loss: 2.1463 - val_accuracy: 0.5387 - val_loss: 1.9805
Epoch 9/12
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5026 - loss: 2.1321
Epoch 9: val_accuracy improved from 0.53873 to 0.54080, saving model to sketchxai_baseline_345_model.h5



Epoch 9: finished saving model to sketchxai_baseline_345_model.h5
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 138s 14ms/step - accuracy: 0.5031 - loss: 2.1315 - val_accuracy: 0.5408 - val_loss: 1.9620
Epoch 10/12
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5061 - loss: 2.1167
Epoch 10: val_accuracy improved from 0.54080 to 0.54120, saving model to sketchxai_baseline_345_model.h5



Epoch 10: finished saving model to sketchxai_baseline_345_model.h5
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 128s 13ms/step - accuracy: 0.5057 - loss: 2.1198 - val_accuracy: 0.5412 - val_loss: 1.9584
Epoch 11/12
10106/10107 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5081 - loss: 2.1085
Epoch 11: val_accuracy improved from 0.54120 to 0.54681, saving model to sketchxai_baseline_345_model.h5



Epoch 11: finished saving model to sketchxai_baseline_345_model.h5
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 140s 14ms/step - accuracy: 0.5077 - loss: 2.1097 - val_accuracy: 0.5468 - val_loss: 1.9401
Epoch 12/12
10104/10107 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5100 - loss: 2.0951
Epoch 12: val_accuracy did not improve from 0.54681
10107/10107 ━━━━━━━━━━━━━━━━━━━━ 130s 13ms/step - accuracy: 0.5096 - loss: 2.0999 - val_accuracy: 0.5447 - val_loss: 1.9460
Restoring model weights from the end of the best epoch: 11.
